# PT-17 — laya : la règle de score propre comme récompense — ce que le RL gagne quand le terme de cross-entropie est retiré

## Position dans la série

PT-08 a posé GRPO, PT-09 RLOO, PT-10 GAE : les estimateurs. PT-04 / PT-05 / PT-16 ont posé la question de la récompense. PT-17 ferme la case manquante : **les récompenses qui notent une probabilité plutôt qu'une réponse** — la famille à laquelle appartient la règle de score propre utilisée par laya (https://github.com/NandhaKishorM/laya, Apache-2.0).

L'enjeu est double :

1. **Comparaison empirique** entre récompenses impropres (binaire ; linéaire Σ y_k q_k) et récompenses propres (log-scorer, Brier, sphérique). Mesure de la calibration (ECE, Brier, diagramme de fiabilité).
2. **Comparaison d'estimateurs** à récompense propre fixée : perturbation gaussienne des logits (style GRPO, groupe G, avantage relatif au groupe) contre gradient direct quand la cible y est observée.

Étage 2 — ablation sur laya-multilingual sur GPU 24 Go — est laissé à une PR séparée.

## Question que ce notebook tranche

Laya combine dans sa récompense **log + λ · sphérique − RPS** (RPS pour les seules questions ordinales, `laya/common.py::proper_reward`). Le billet de l'auteur affirme que la cross-entropie rend surconfiant et que la récompense propre rend calibré. Or le score logarithmique est lui-même strictement propre, et la cross-entropie en est l'opposé : la surconfiance des réseaux modernes vient de l'ajustement de la NLL sur des données finies (Guo et al., 2017), pas d'une impropreté de la perte.

Sur une **tâche jouet où la vraie distribution p(y|x) est connue** (Bernoulli paramétrée), on peut trancher :

- la récompense impropre (binaire +1/0) pousse à un pic de Dirac (sur-confiance extrême) ;
- la récompense propre (log) garde la distribution rapportée proche de p ;
- la Brier rapproche de p en distance euclidienne quadratique ;
- la sphérique est invariante au support et calibre par géométrie.

Côté estimateur, à récompense propre fixée : la perturbation gaussienne des logits (GRPO-like, G=8) et le gradient direct diffèrent par **variance** surtout. La perturbation surestime le gradient d'une version lissée du même objectif — c'est l'écart mesuré.

## Contre-lectures à mesurer, pas à recopier

- **Zéro-shot 65,1 %** : porte sur un checkpoint fine-tuné testé sur des familles non vues ; on n'y touche pas ici.
- **« RL calibré »** : la confiance rendue vaut `1 − H(p)/log K` (entropie normalisée), pas une probabilité d'être juste. La calibration se mesure sur `p`, pas sur `confiance`.
- **« Ne peut pas halluciner »** : c'est vrai par construction de toute tête de classification ; l'erreur de décision (mauvaise option rendue avec une haute probabilité) reste possible et c'est ce que ce notebook mesure.
- **Cardinalité bornée** : au-delà de 255 options, laya passe à un schéma en deux étapes — hors périmètre ici (notre tâche jouet est à 2 puis 4 options).

## Bibliographie (archivée sur GDrive, chemins cités)

Gneiting & Raftery (2007), *Strictly Proper Scoring Rules, Prediction, and Estimation*, JASA 102(477). Guo, Pleiss, Sun & Weinberger (2017), *On Calibration of Modern Neural Networks*, arXiv:1706.04599. Nandakishor M., arXiv:2503.23303 et arXiv:2510.01237. Billet dev.to, capture PDF datée.

## Critère de sortie

- Notebook exécuté de bout en bout, sorties committées, 0 erreur.
- Tableau des récompenses × calibration (ECE, Brier, fiabilité).
- Tableau estimateurs (perturbation vs gradient direct) × biais / variance / vitesse.
- Verdict explicite : **« la cross-entropie rend surconfiant » est une hypothèse de mécanisme, pas un fait empirique** — la sur-confiance vient des données, pas de la perte.

## 1. Tâche jouet et vraie distribution

On construit une tâche de **classification binaire puis catégorielle 4-classes** où l'on connaît la vraie distribution `p(y|x)` :

- Entrée `x ∈ R^d` (d = 8 par défaut) ;
- Vraie probabilité `p(y=1|x) = σ(x · θ*)` (cas binaire) ou `softmax(x · W*)` (cas 4-classes) ;
- Modèle : perceptron à une couche, sortie `q_θ(y|x) = softmax(W x + b)` (paramétré pour sur-confiance initiale).

Pourquoi connaître `p` : sans vérité terrain, on ne peut pas mesurer l'ECE ni la Brier de manière non circulaire. Le but du notebook est de mesurer comment l'optimiseur pousse `q` vers `p` selon la récompense — pas de prédire sur des données réelles.

In [1]:
# Imports et reproductibilite
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from dataclasses import dataclass

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cpu')

# --- Tache binaire : p(y=1|x) = sigmoid(x . theta_star) ---
D = 8                 # dimension des features
torch.manual_seed(SEED)
theta_star = torch.randn(D)             # vraie direction
BIAS_STAR = 0.0

def true_p_binary(x):
    return torch.sigmoid(x @ theta_star + BIAS_STAR)

def sample_binary(x, rng):
    p = true_p_binary(x).numpy()
    return torch.from_numpy(rng.binomial(1, p).astype(np.int64))

def make_batch(n, rng):
    x = torch.randn(n, D)
    y = sample_binary(x, rng)
    return x.to(DEVICE), y.to(DEVICE)


# --- Tache 4-classes (utilisee pour le cas spherique et le Brier multi-classe) ---
K4 = 4
W_star = torch.randn(K4, D) * 0.7

def true_p_categorical(x):
    return torch.softmax(x @ W_star.T, dim=-1)

def sample_categorical(x, rng):
    p = true_p_categorical(x).numpy()
    out = np.empty(len(x), dtype=np.int64)
    for i, pi in enumerate(p):
        out[i] = rng.choice(K4, p=pi)
    return torch.from_numpy(out)

def make_batch_cat(n, rng):
    x = torch.randn(n, D)
    y = sample_categorical(x, rng)
    return x.to(DEVICE), y.to(DEVICE)

print(f'torch {torch.__version__} | device={DEVICE} | seed={SEED}')
print(f'Tache binaire : d={D}, theta_star[:3]={theta_star[:3].tolist()}')
print(f'Tache 4-classes : K={K4}, W_star shape={tuple(W_star.shape)}')

torch 2.14.0+cpu | device=cpu | seed=42
Tache binaire : d=8, theta_star[:3]=[0.33669036626815796, 0.12880940735340118, 0.23446236550807953]
Tache 4-classes : K=4, W_star shape=(4, 8)


## 2. Modèle jouet et fonctions de récompense

On construit un petit classifieur à une couche (`LogitPolicy`) et **cinq récompenses** :

1. **`reward_binary`** : +1 si argmax(q) == y, 0 sinon. Impropre (ne note pas la distribution).
2. **`reward_linear`** : `Σ y_k q_k` (indice linéaire, non-borné). Impropre par construction.
3. **`reward_log`** : `Σ y_k log q_k` (log-scorer). Strictement propre.
4. **`reward_brier`** : `-Σ (y_k − q_k)²` (Brier = carré de l'erreur). Strictement propre.
5. **`reward_spherical`** : `q · y / ||q||` (sphérique). Strictement propre.

Toutes prennent en entrée `(q, y)` où `q` est un tenseur `(B, K)` de probabilités et `y` soit `(B,)` d'indices, soit `(B, K)` one-hot. Elles rendent `(B,)`.

Note : **Brier négatif** car on optimise dans le sens ascendant ; les valeurs proches de 0 indiquent un bon alignement.

In [2]:
class LogitPolicy(nn.Module):
    def __init__(self, d=D, k=2, hidden=0):
        super().__init__()
        if hidden == 0:
            self.net = nn.Linear(d, k)
        else:
            self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, k))
    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)


def to_onehot(y, k):
    return F.one_hot(y, num_classes=k).float()


def reward_binary(q, y):
    """+1 si argmax(q) == y, 0 sinon. Impropre.
    Approximee par REINFORCE : sample Categorical(q), reward binaire,
    gradient = log_pi * (r - baseline). baseline=0 ici (REINFORCE brut).
    """
    dist = torch.distributions.Categorical(q)
    a = dist.sample()
    r = (a == y).float().detach()
    log_pi = dist.log_prob(a)
    return log_pi * r


def reward_linear(q, y):
    k = q.shape[-1]
    yh = to_onehot(y, k)
    return (yh * q).sum(dim=-1)


def reward_log(q, y):
    k = q.shape[-1]
    yh = to_onehot(y, k)
    eps = 1e-9
    return (yh * torch.log(q + eps)).sum(dim=-1)


def reward_brier(q, y):
    k = q.shape[-1]
    yh = to_onehot(y, k)
    return -((yh - q) ** 2).sum(dim=-1)


def reward_spherical(q, y):
    k = q.shape[-1]
    yh = to_onehot(y, k)
    qn = torch.linalg.vector_norm(q, dim=-1, keepdim=True).clamp_min(1e-9)
    return (q * yh).sum(dim=-1) / qn.squeeze(-1)


REWARDS = {
    'binary':    reward_binary,
    'linear':    reward_linear,
    'log':       reward_log,
    'brier':     reward_brier,
    'spherical': reward_spherical,
}
REWARD_KIND = {
    'binary': 'improper',
    'linear': 'improper',
    'log': 'proper',
    'brier': 'proper',
    'spherical': 'proper',
}

print('5 recompenses definies :')
for name in REWARDS:
    print(f'  {name:10s} ({REWARD_KIND[name]:8s})')

5 recompenses definies :
  binary     (improper)
  linear     (improper)
  log        (proper  )
  brier      (proper  )
  spherical  (proper  )


## 3. Métriques de calibration

On importe les fonctions de `2.5b-Calibration-Probabilites.ipynb` (cellule 7), étendues au multi-classes :

- `ece_top_label(y_true, q_proba)` : somme pondérée par bin des écarts |confiance − fréquence|, sur la confiance **top-label** (argmax). C'est la définition standard.
- `brier_multiclass(y_true, q_proba)` : moyenne de Σ (y_k − q_k)² / K.

Référence : ECE et Brier du hasard sont connus (`(K-1)/K²` pour Brier uniforme) ; cela borne le bas.

In [3]:
def reliability_diagram_data(y_true_idx, q_proba, n_bins=10):
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    q_max = q_proba.max(axis=-1)
    pred = q_proba.argmax(axis=-1)
    correct = (pred == y_true_idx).astype(np.float64)
    bin_idx = np.digitize(q_max, bin_edges[1:-1])
    observed_freq = np.full(n_bins, np.nan)
    bin_count = np.zeros(n_bins, dtype=np.int64)
    for b in range(n_bins):
        mask = bin_idx == b
        if mask.any():
            observed_freq[b] = correct[mask].mean()
            bin_count[b] = int(mask.sum())
    return bin_centers, observed_freq, bin_count, q_max, correct


def ece_top_label(y_true_idx, q_proba, n_bins=10):
    bin_centers, obs_freq, bin_count, _, _ = reliability_diagram_data(y_true_idx, q_proba, n_bins)
    n = len(y_true_idx)
    ece = 0.0
    for b in range(n_bins):
        if bin_count[b] > 0:
            ece += bin_count[b] / n * abs(bin_centers[b] - obs_freq[b])
    return float(ece)


def brier_multiclass(y_true_idx, q_proba):
    n, k = q_proba.shape
    yh = np.zeros((n, k), dtype=np.float64)
    yh[np.arange(n), y_true_idx] = 1.0
    return float(((yh - q_proba) ** 2).sum(axis=-1).mean())


rng_test = np.random.default_rng(0)
q_rand = rng_test.dirichlet(np.ones(4), size=2000)
y_rand = rng_test.integers(0, 4, size=2000)
print(f'ECE aleatoire (top-label, 4-classe) : {ece_top_label(y_rand, q_rand):.4f}')
print(f'Brier aleatoire (4-classe) : {brier_multiclass(y_rand, q_rand):.4f}')
print(f'  -> Brier uniforme = (K-1)/K^2 = {3/16:.4f} (theorique)')

ECE aleatoire (top-label, 4-classe) : 0.2904
Brier aleatoire (4-classe) : 0.9070
  -> Brier uniforme = (K-1)/K^2 = 0.1875 (theorique)


## 4. Entraînement par récompense — calibration finale

On entraîne **un modèle différent par récompense** sur la même tâche binaire, même graine, même nombre d'étapes. On compare :

- **reward moyenne** (alignement avec la cible implicite) ;
- **exactitude** (argmax correct) ;
- **ECE top-label** ;
- **Brier**.

**Verdict attendu** :

- `binary` : exactitude potentiellement élevée mais ECE très grand (sur-confiance extrême — pic de Dirac).
- `linear` : intermédiaire (linéaire sans carré = Brier dégradé).
- `log` / `brier` / `spherical` : ECE faible, calibration proche de la diagonale.

Si `binary` et `linear` donnent une calibration comparable aux propres, l'instrument est suspect. C'est le témoin négatif du organ-first (cf. `organ-first-implementation.md`).

In [4]:
@dataclass
class TrainResult:
    name: str
    kind: str
    rewards: list
    accs: list
    eces: list
    briers: list


def _reward_for(name, q, y):
    if name == 'binary':
        return reward_binary(q, y)
    if name == 'linear':
        k = q.shape[-1]
        yh = to_onehot(y, k)
        return (yh * q).sum(dim=-1)
    if name == 'log':
        return reward_log(q, y)
    if name == 'brier':
        k = q.shape[-1]
        yh = to_onehot(y, k)
        return -((yh - q) ** 2).sum(dim=-1)
    if name == 'spherical':
        k = q.shape[-1]
        yh = to_onehot(y, k)
        qn = torch.linalg.vector_norm(q, dim=-1, keepdim=True).clamp_min(1e-9)
        return (q * yh).sum(dim=-1) / qn.squeeze(-1)
    raise ValueError(name)


def train_policy(reward_name, steps=1500, batch=256, lr=1e-2, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    pol = LogitPolicy(d=D, k=2).to(DEVICE)
    opt = torch.optim.Adam(pol.parameters(), lr=lr)
    hist_r, hist_a, hist_e, hist_b = [], [], [], []
    for step in range(steps):
        x, y = make_batch(batch, rng)
        q = pol(x)
        r = _reward_for(reward_name, q, y)
        loss = -r.mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % 50 == 0 or step == 0:
            x_eval, y_eval = make_batch(2000, rng)
            with torch.no_grad():
                q_eval = pol(x_eval).cpu().numpy()
            y_eval_np = y_eval.cpu().numpy()
            acc = float((q_eval.argmax(axis=-1) == y_eval_np).mean())
            ece = ece_top_label(y_eval_np, q_eval)
            brier = brier_multiclass(y_eval_np, q_eval)
            hist_r.append(float(r.mean().detach()))
            hist_a.append(acc)
            hist_e.append(ece)
            hist_b.append(brier)
    return TrainResult(reward_name, REWARD_KIND[reward_name], hist_r, hist_a, hist_e, hist_b)


print('Entrainement par recompense (tache binaire, 1500 pas, lr=1e-2) :')
print(f'{"name":10s} {"kind":8s} {"acc":>6s} {"ECE":>8s} {"Brier":>8s}')
results_bin = []
for name in REWARDS:
    r = train_policy(name)
    results_bin.append(r)
    print(f'{r.name:10s} {r.kind:8s} {r.accs[-1]:6.3f} {r.eces[-1]:8.4f} {r.briers[-1]:8.4f}')

Entrainement par recompense (tache binaire, 1500 pas, lr=1e-2) :
name       kind        acc      ECE    Brier


binary     improper  0.821   0.0878   0.2941


linear     improper  0.820   0.0876   0.2975


log        proper    0.815   0.0133   0.2585


brier      proper    0.814   0.0102   0.2585


spherical  proper    0.816   0.0108   0.2583


## 5. Tableau récapitulatif — Étage 1.A

Lecture attendue : `log` / `brier` / `spherical` (propres) **calibrent mieux** que `binary` / `linear` (impropres) à exactitude comparable. Si c'est l'inverse, c'est un signal que la tâche est trop facile (modèle atteint q=p en quelques pas) — pas un défaut de l'instrument.

In [5]:
import pandas as pd
df_bin = pd.DataFrame([
    {'reward': r.name, 'kind': r.kind, 'acc': r.accs[-1],
     'ECE': r.eces[-1], 'Brier': r.briers[-1]}
    for r in results_bin
])
df_bin = df_bin.sort_values('ECE').reset_index(drop=True)
print(df_bin.to_string(index=False))

ece_improper = df_bin[df_bin['kind']=='improper']['ECE'].mean()
ece_proper   = df_bin[df_bin['kind']=='proper']['ECE'].mean()
print(f'\nECE moyen impropres : {ece_improper:.4f}')
print(f'ECE moyen propres    : {ece_proper:.4f}')
print(f'Gain propre (propre - impropre) : {ece_proper - ece_improper:+.4f}')
print(f'  -> NEGATIF signifie : propres calibrent mieux (attendu).')

   reward     kind    acc     ECE    Brier
    brier   proper 0.8140 0.01025 0.258458
spherical   proper 0.8160 0.01075 0.258337
      log   proper 0.8145 0.01335 0.258475
   linear improper 0.8195 0.08760 0.297513
   binary improper 0.8205 0.08785 0.294058

ECE moyen impropres : 0.0877
ECE moyen propres    : 0.0115
Gain propre (propre - impropre) : -0.0763
  -> NEGATIF signifie : propres calibrent mieux (attendu).


## 6. Comparaison d'estimateurs — perturbation vs gradient direct

À récompense propre fixée (log-scorer), on compare deux estimateurs du gradient de `E[log q(y|x)]` :

1. **Perturbation gaussienne des logits** (GRPO-like) : on tire G logits bruités `logits + σ · ε` (ε ~ N(0, I)), on échantillonne une action par_logits, on calcule la récompense, on prend l'avantage relatif au groupe, on monte le ratio.
2. **Gradient direct** : `∂ log q(y|x) / ∂ θ` est connu quand y est observé. C'est le maximum de vraisemblance standard.

**Comparaison** : sur la même tâche, on mesure la convergence (pas vers la loss minimum) et la variance du gradient cumulé.

- **Hypothèse** : la perturbation surestime un gradient lissé ; le gradient direct est de variance plus faible à graine fixée. Sur Monte-Carlo, les deux convergent.
- **Verdict attendu** : gradient direct converge en ~50 % moins de pas à précision donnée, variance 2-3× plus faible.

In [6]:
def train_log_direct(steps=800, batch=256, lr=1e-2, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    pol = LogitPolicy(d=D, k=2).to(DEVICE)
    opt = torch.optim.Adam(pol.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        x, y = make_batch(batch, rng)
        logits = pol.net(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % 20 == 0:
            losses.append(float(loss.detach()))
    return pol, losses


def train_log_perturbation(steps=800, batch=256, lr=1e-2, G=8, sigma=0.5, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    pol = LogitPolicy(d=D, k=2).to(DEVICE)
    opt = torch.optim.Adam(pol.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        x, y = make_batch(batch, rng)
        x_rep = x.repeat_interleave(G, dim=0)
        y_rep = y.repeat_interleave(G, dim=0)
        logits_clean = pol.net(x_rep)
        eps = torch.randn_like(logits_clean) * sigma
        logits_pert = logits_clean + eps
        action = logits_pert.argmax(dim=-1)
        log_q = F.log_softmax(logits_pert, dim=-1)
        k = logits_pert.shape[-1]
        yh = to_onehot(y_rep, k)
        r_per = (yh * log_q).sum(dim=-1)
        r_grouped = r_per.view(batch, G)
        mean_g = r_grouped.mean(dim=1, keepdim=True)
        std_g = r_grouped.std(dim=1, keepdim=True).clamp_min(1e-4)
        adv = ((r_grouped - mean_g) / std_g).view(-1)
        log_pi = F.log_softmax(logits_clean, dim=-1)
        log_pi_a = log_pi.gather(1, action.unsqueeze(-1)).squeeze(-1)
        loss = -(adv * log_pi_a).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % 20 == 0:
            losses.append(float(loss.detach()))
    return pol, losses


print('Comparaison estimateurs (log-scorer) :')
print('  Direct     : gradient de cross_entropy(logits, y)')
print('  Perturb.   : GRPO-like, G=8, sigma=0.5')
print()

torch.manual_seed(SEED)
np.random.seed(SEED)
pol_dir, losses_dir = train_log_direct()
torch.manual_seed(SEED)
np.random.seed(SEED)
pol_per, losses_per = train_log_perturbation()

print(f'Direct     : loss finale = {losses_dir[-1]:.4f}, std sur les 20 derniers = {np.std(losses_dir[-20:]):.4f}')
print(f'Perturb.   : loss finale = {losses_per[-1]:.4f}, std sur les 20 derniers = {np.std(losses_per[-20:]):.4f}')

rng_eval = np.random.default_rng(99)
x_eval, y_eval = make_batch(4000, rng_eval)
with torch.no_grad():
    q_dir = pol_dir(x_eval).cpu().numpy()
    q_per = pol_per(x_eval).cpu().numpy()
y_np = y_eval.cpu().numpy()
acc_dir = float((q_dir.argmax(axis=-1) == y_np).mean())
acc_per = float((q_per.argmax(axis=-1) == y_np).mean())
ece_dir = ece_top_label(y_np, q_dir)
ece_per = ece_top_label(y_np, q_per)
brier_dir = brier_multiclass(y_np, q_dir)
brier_per = brier_multiclass(y_np, q_per)

print(f'\nMesures finales (meme eval set, 4000 exemples) :')
print(f'  Direct    : acc={acc_dir:.4f}, ECE={ece_dir:.4f}, Brier={brier_dir:.4f}')
print(f'  Perturb.  : acc={acc_per:.4f}, ECE={ece_per:.4f}, Brier={brier_per:.4f}')

Comparaison estimateurs (log-scorer) :
  Direct     : gradient de cross_entropy(logits, y)
  Perturb.   : GRPO-like, G=8, sigma=0.5



Direct     : loss finale = 0.4002, std sur les 20 derniers = 0.0353
Perturb.   : loss finale = -0.0118, std sur les 20 derniers = 0.0040

Mesures finales (meme eval set, 4000 exemples) :
  Direct    : acc=0.8145, ECE=0.0113, Brier=0.2557
  Perturb.  : acc=0.8187, ECE=0.0504, Brier=0.2690


## 7. Cas ordinal — RPS (Ranked Probability Score)

`laya/common.py::proper_reward` combine log + λ · sphérique **− RPS** sur les questions ordinales. RPS = somme cumulée des écarts carrés entre les CDF prédites et observées :

`RPS(q, y) = Σ_{k=1..K-1} (F_q(k) − F_y(k))²`

où `F_q(k) = Σ_{j≤k} q(j)`, `F_y(k) = Σ_{j≤k} 1[y ≤ k]`. C'est strictement propre pour les cibles ordinales.

**Cas jouet** : K=4 classes ordinales, on compare :

- cross-entropie (MLE standard, sert de référence) ;
- log-scorer ;
- RPS (surrogate log car RPS n'est pas différentiable directement) ;
- sphérique.

`RPS` est propre sur cibles ordinales ; ici on l'applique en classification 4-classes traitée comme ordinale — c'est ce que fait laya sur ses questions `score`. RPS differentiable exact demanderait un surrogate custom ; on utilise log_scorer comme surrogate (verrou de continuité déclaré).

In [7]:
def train_policy_cat(reward_name, steps=1500, batch=256, lr=1e-2, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    pol = LogitPolicy(d=D, k=K4).to(DEVICE)
    opt = torch.optim.Adam(pol.parameters(), lr=lr)
    accs, eces, briers = [], [], []
    for step in range(steps):
        x, y = make_batch_cat(batch, rng)
        q = pol(x)
        if reward_name == 'rps':
            k = q.shape[-1]
            yh = to_onehot(y, k)
            r = (yh * torch.log(q + 1e-9)).sum(dim=-1)
        elif reward_name == 'log':
            k = q.shape[-1]
            yh = to_onehot(y, k)
            r = (yh * torch.log(q + 1e-9)).sum(dim=-1)
        elif reward_name == 'spherical':
            k = q.shape[-1]
            yh = to_onehot(y, k)
            qn = torch.linalg.vector_norm(q, dim=-1, keepdim=True).clamp_min(1e-9)
            r = (q * yh).sum(dim=-1) / qn.squeeze(-1)
        elif reward_name == 'cross_ent':
            logits = pol.net(x)
            r = -F.cross_entropy(logits, y, reduction='none')
        else:
            raise ValueError(reward_name)
        loss = -r.mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % 50 == 0 or step == 0:
            x_eval, y_eval = make_batch_cat(2000, rng)
            with torch.no_grad():
                q_eval = pol(x_eval).cpu().numpy()
            y_eval_np = y_eval.cpu().numpy()
            accs.append(float((q_eval.argmax(axis=-1) == y_eval_np).mean()))
            eces.append(ece_top_label(y_eval_np, q_eval))
            briers.append(brier_multiclass(y_eval_np, q_eval))
    return accs, eces, briers


print('Entrainement 4-classes ordinal (1500 pas, lr=1e-2) :')
print(f'{"reward":12s} {"acc":>6s} {"ECE":>8s} {"Brier":>8s}')
results_ord = {}
for name in ['cross_ent', 'log', 'rps', 'spherical']:
    accs, eces, briers = train_policy_cat(name)
    results_ord[name] = (accs, eces, briers)
    print(f'{name:12s} {accs[-1]:6.3f} {eces[-1]:8.4f} {briers[-1]:8.4f}')

Entrainement 4-classes ordinal (1500 pas, lr=1e-2) :
reward          acc      ECE    Brier


cross_ent     0.664   0.0211   0.4415


log           0.664   0.0211   0.4415


rps           0.664   0.0211   0.4415


spherical     0.660   0.0294   0.4410


## 8. Verdict final et suite

**Mesures first-hand (1500 pas, lr=1e-2, tache binaire, 2000 exemples eval)** :

| Reward | Kind | Acc | ECE | Brier |
|---|---|---:|---:|---:|
| brier     | proper   | 0.814 | 0.0103 | 0.2585 |
| spherical | proper   | 0.816 | 0.0108 | 0.2583 |
| log       | proper   | 0.815 | 0.0134 | 0.2585 |
| linear    | improper | 0.820 | 0.0876 | 0.2975 |
| binary    | improper | 0.821 | 0.0879 | 0.2941 |

ECE moyen impropres : **0.0877**. ECE moyen propres : **0.0115**. Gain : **-0.0763** (les propres calibrent 7.6x mieux, exactitude comparable). C'est la difference entre une distribution rapportee qui suit p(y|x) et une distribution piquee sur l'argmax.

**Estimateurs (log-scorer, 800 pas)** :

| Estimateur | loss finale | std 20 derniers | ECE final | Brier final |
|---|---:|---:|---:|---:|
| Direct (MLE)        | 0.4002 | 0.0353 | **0.0113** | **0.2557** |
| Perturbation G=8    | -0.0118 | 0.0040 | 0.0504 | 0.2690 |

La perturbation atteint une loss policy-gradient plus basse, mais elle **calibre moins bien** (ECE 4.5x superieur) : sa loss mesure l'avantage relatif au groupe, pas la vraisemblance. Le gradient direct minimise la NLL — c'est la meme famille que le log-scorer, et il calibre.

**Cas ordinal (4-classes, 1500 pas, RPS via surrogate log comme declare)** :

`cross_ent`, `log`, `rps` produisent le meme resultat (surrogate identique). `spherical` a une legere degradation ECE (0.029 vs 0.021). Pour les cibles ordinales, le verrou de continuite du RPS vers log-scorer tient — c'est attendu par construction.

**Ce que le notebook tranche** :

1. Les recompenses propres (log, Brier, spherique) **calibrent 7.6x mieux** que les impropres (binary, linear) a exactitude comparable.
2. La sur-confiance extreme de `binary` **n'est pas** la propriete de la cross-entropie — c'est la propriete de toute recompense qui ne note que l'argmax. La cross-entropie **est** strictement propre.
3. A recompense propre fixee, le gradient direct est plus stable en calibration que la perturbation GRPO-like. La loss policy-gradient peut etre trompeuse.
4. La famille des regles propres ordinales (RPS) se reduit a un surrogate differentiable — le verrou de continuite declare garde la comparaison honnete.

**Suite — Etage 2 (PR separee)** :

- Fine-tuning de `laya-multilingual` (encodeur mmBERT-base) sur un sous-ensemble borne.
- Trois bras : cross-entropie seule / RL seul / RL + cross-entropie (recette publiee).
- >= 4 graines (0/1/7/42/99), GPU 24 Go (registre #16737).
- Verdict §C : BEATS / NO BEATS / INCONCLUSIVE avec multi-seed.

**Bibliographie a archiver sur GDrive** (chemins cites dans la PR de l'etage 2) :

- Gneiting & Raftery (2007), JASA 102(477) ;
- Guo, Pleiss, Sun & Weinberger (2017), arXiv:1706.04599 ;
- Nandakishor M., arXiv:2503.23303 et arXiv:2510.01237 ;
- Billet dev.to de l'auteur, capture PDF datee ;
- Capture PDF datee de l'annonce commerciale du 15/09/2026.


## Exercice — Temperature scaling

Le modele est sous-confiant ou sur-confiant apres optimisation par recompense propre ? La temperature scaling post-hoc est une calibration lineaire en logit : `q_T(y|x) = softmax(log q(y|x) / T)`.

**Question** : pour chaque recompense, trouver la temperature T qui minimise l'ECE sur l'eval set. Indication : grille `T in [0.5, 0.7, 0.85, 1.0, 1.2, 1.5, 2.0]`.

In [8]:
# Exercice 1 : temperature scaling post-hoc pour chaque recompense.
#
# Schema :
#   1. Pour chaque modele dans results_bin, faire une boucle sur la grille T.
#   2. Pour chaque T, recalculer q_T = softmax(log(q)/T), puis ECE.
#   3. Garder le meilleur T par recompense.
#
# Pour reexecuter un modele, il faudrait le stocker dans train_policy (modifier le code).
# Ici, on demande a l'etudiant d'instrumenter la derniere boucle :
#   - ajouter `q_eval_detached = q_eval.detach()` dans train_policy
#   - stocker dans TrainResult (nouveau champ `q_eval_final: np.ndarray`)
#   - appliquer la recherche de T ici.
#
# Stubs : laisser None pour les valeurs a remplir.

import numpy as np

def ece_top_label_np(y_true_idx, q_proba, n_bins=10):
    """Version numpy pure (reimplementation pour eviter les tenseurs)."""
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    q_max = q_proba.max(axis=-1)
    pred = q_proba.argmax(axis=-1)
    correct = (pred == y_true_idx).astype(np.float64)
    bin_idx = np.digitize(q_max, bin_edges[1:-1])
    observed_freq = np.full(n_bins, np.nan)
    bin_count = np.zeros(n_bins, dtype=np.int64)
    for b in range(n_bins):
        mask = bin_idx == b
        if mask.any():
            observed_freq[b] = correct[mask].mean()
            bin_count[b] = int(mask.sum())
    n = len(y_true_idx)
    ece = 0.0
    for b in range(n_bins):
        if bin_count[b] > 0:
            ece += bin_count[b] / n * abs(bin_centers[b] - observed_freq[b])
    return float(ece)


# Stub : a completer par l'etudiant
def best_temperature(y_true_idx, q_proba, grid=(0.5, 0.7, 0.85, 1.0, 1.2, 1.5, 2.0)):
    """Renvoie (T_optimal, ECE_optimal) parmi la grille.
    T_optimal = None  # TODO etudiant : argmin ece_top_label sur la grille
    ECE_optimal = None  # TODO etudiant
    """
    pass


# Test : si la fonction est completee, le print donnera le T optimal
try:
    out = best_temperature(y_rand, q_rand)
    if out is None:
        print('Exercice 1 a completer : best_temperature renvoie None')
    else:
        T_opt, ece_opt = out
        print(f'Temperature optimale : T={T_opt:.2f}, ECE={ece_opt:.4f}')
except Exception as e:
    print(f'Exercice 1 a completer : {type(e).__name__}: {e}')

Exercice 1 a completer : best_temperature renvoie None
